# 탐색적 데이터 분석 (EDA) 및 데이터 전처리

이 노트북에서는 데이터의 기본 통계량, 분포, 결측치, 이상치 등을 분석하고,
머신러닝 모델에 적합하도록 데이터를 전처리합니다.

In [ ]:
# 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
df = pd.read_csv('data/raw_data.csv')
print(f'데이터 크기: {df.shape}')
print(f'\n컬럼 정보:')
print(df.info())
df.head()

In [ ]:
# 기본 통계량
print('=' * 50)
print('기본 통계량')
print('=' * 50)
df.describe()

In [ ]:
# 결측치 확인
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print('결측치 현황:')
    print(missing_df)
    
    plt.figure(figsize=(10, 6))
    missing_df['Missing Percentage'].plot(kind='barh')
    plt.title('컬럼별 결측치 비율')
    plt.xlabel('결측치 비율 (%)')
    plt.tight_layout()
    plt.show()
else:
    print('결측치가 없습니다.')

In [ ]:
# 수치형 변수 분포 확인
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 0:
    n_cols = 3
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes
    
    for idx, col in enumerate(numeric_cols):
        if idx < len(axes):
            df[col].hist(bins=30, ax=axes[idx], edgecolor='black')
            axes[idx].set_title(f'{col} 분포')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('빈도')
    
    for idx in range(len(numeric_cols), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 상관관계 분석
if len(numeric_cols) > 1:
    corr_matrix = df[numeric_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={'shrink': .8}, fmt='.2f')
    plt.title('수치형 변수 간 상관관계 행렬')
    plt.tight_layout()
    plt.show()
    
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
    
    if high_corr_pairs:
        print('\n높은 상관관계를 가진 변수 쌍 (|r| > 0.7):')
        for pair in high_corr_pairs:
            print(f'{pair[0]} <-> {pair[1]}: {pair[2]:.3f}')

In [ ]:
# 이상치 탐지 (IQR 방법)
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

outlier_summary = []
for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_summary.append({
        'Column': col,
        'Outlier Count': len(outliers),
        'Outlier Percentage': (len(outliers) / len(df)) * 100,
        'Lower Bound': lower,
        'Upper Bound': upper
    })

outlier_df = pd.DataFrame(outlier_summary)
print('이상치 탐지 결과:')
print(outlier_df)

In [ ]:
# 데이터 전처리
df_processed = df.copy()

# 결측치 처리: 수치형 변수는 중앙값으로 대체
for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        median_value = df_processed[col].median()
        df_processed[col].fillna(median_value, inplace=True)
        print(f'{col}: {df_processed[col].isnull().sum()}개의 결측치를 중앙값({median_value:.2f})으로 대체')

# 범주형 변수: 최빈값으로 대체
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        mode_value = df_processed[col].mode()[0] if len(df_processed[col].mode()) > 0 else 'Unknown'
        df_processed[col].fillna(mode_value, inplace=True)
        print(f'{col}: {df_processed[col].isnull().sum()}개의 결측치를 최빈값({mode_value})으로 대체')

In [ ]:
# 범주형 변수 인코딩
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col + '_encoded'] = le.fit_transform(df_processed[col].astype(str))
    label_encoders[col] = le
    print(f'{col} 인코딩 완료: {len(le.classes_)}개 클래스')

In [ ]:
# 수치형 변수 스케일링
scaler = StandardScaler()
df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])
print('수치형 변수 표준화 완료')
print(f'스케일링된 변수: {numeric_cols}')

In [ ]:
# 전처리된 데이터 저장
df_processed.to_csv('data/processed_data.csv', index=False)
print('전처리된 데이터가 \'data/processed_data.csv\'에 저장되었습니다.')